# Couverture d'options par apprentissage par renforcement — recherche

**Portage du chapitre 07-01 du livre *Hands-On AI Trading*** ( Better Hedging with Reinforcement Learning ) — issue [#18902](https://github.com/jsboige/CoursIA/issues/18902).

**Objectif.** Un agent RL apprend à couvrir une position courte d'un call européen ATM de 30 jours sur SPY, et on compare son P&L de couverture à celui de la couverture delta de Black-Scholes quotidienne — la référence du marché. La mesure porte sur la **variance** et la **queue (CVaR 95 %)** du P&L de couverture, avec et sans frais, sur des régimes de volatilité contrastés (calme, 2020, 2022).

**Plan** (celui du README du projet) : simulation GBM, environnement gymnasium (état : prix normalisé, temps restant, delta et gamma BS, position courante ; action : ratio de couverture continu dans [0, 1] ; récompense : P&L terminal de couverture), entraînement PPO multi-seed, évaluation contre la baseline delta.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

import gymnasium as gym
from gymnasium import spaces

RNG_SEED = 18902
np.random.seed(RNG_SEED)
print("numpy", np.__version__, "| gymnasium", gym.__version__)

numpy 2.3.5 | gymnasium 1.3.0


## 1. Pricing Black-Scholes

Call européen, sans dividende (SPY versé en continu serait un raffinement ; le livre travaille aussi sans dividende). On utilise $S$ le prix du sous-jacent, $K$ le strike, $\sigma$ la volatilité, $r = 0$ le taux (simplification documentée : l'objectif compare deux politiques de couverture sur les *mêmes* trajectoires, le niveau exact de $r$ ne change pas le classement).

In [2]:
def bs_call_price(S, K, sigma, tau):
    """Prix du call europeen (r=0, q=0)."""
    tau = np.maximum(tau, 1e-8)
    sig = np.maximum(sigma, 1e-8)
    d1 = (np.log(S / K) + 0.5 * sig * sig * tau) / (sig * np.sqrt(tau))
    d2 = d1 - sig * np.sqrt(tau)
    return S * norm.cdf(d1) - K * norm.cdf(d2)


def bs_delta(S, K, sigma, tau):
    tau = np.maximum(tau, 1e-8)
    sig = np.maximum(sigma, 1e-8)
    d1 = (np.log(S / K) + 0.5 * sig * sig * tau) / (sig * np.sqrt(tau))
    return norm.cdf(d1)


def bs_gamma(S, K, sigma, tau):
    tau = np.maximum(tau, 1e-8)
    sig = np.maximum(sigma, 1e-8)
    d1 = (np.log(S / K) + 0.5 * sig * sig * tau) / (sig * np.sqrt(tau))
    return norm.pdf(d1) / (S * sig * np.sqrt(tau))


# Temoin : call ATM 30 j, vol 20 % -- prix ~ 2,27 % du spot, delta ~ 0,52
S0, K0, sig0, T0 = 100.0, 100.0, 0.20, 30 / 365
print(f"prix={bs_call_price(S0, K0, sig0, T0):.4f}  "
      f"delta={bs_delta(S0, K0, sig0, T0):.4f}  "
      f"gamma={bs_gamma(S0, K0, sig0, T0):.6f}")

prix=2.2872  delta=0.5114  gamma=0.069548


## 2. Simulation des trajectoires — trois régimes de volatilité

GBM quotidien **sous mesure physique** ($\mu = 8\ \%$/an : `DRIFT`). Pourquoi pas la mesure neutre au risque ($\mu = 0$) ? Sous $\mu = 0$, $E[P\&L] = 0$ pour **toute** politique admissible (martingale) : le gradient de politique s'annule en espérance et PPO n'a aucun signal à suivre — mesuré sur ce notebook même : 4 seeds sur 8 s'effondrent sur la même politique non apprise (std ~4,2 sur le run neutre écarté, non conservé dans les sorties committées, pire que non couverte). Le livre de référence entraîne de la même façon contre des données réelles dérivées (mesure P). La **comparaison** est inchangée : les deux politiques jouent les mêmes trajectoires. Trois régimes calés sur les périodes demandées par l'issue :

| Régime | Vol annualisée | Analogue historique |
|---|---|---|
| `calm` | 12 % | période calme (ex. 2017) |
| `covid` | 35 % | mars 2020 |
| `inflation` | 25 % | 2022 |

Chaque trajectoire : 21 séances (30 jours calendaires), pas quotidien $dt = 1/252$.

In [3]:
TRADING_DAYS = 21
DT = 1.0 / 252.0
REGIMES = {"calm": 0.12, "covid": 0.35, "inflation": 0.25}
DRIFT = 0.08  # mesure P (derive 8 %/an) -- voir section 2


def simulate_paths(n_paths, sigma, seed, S0=100.0, mu=0.0):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n_paths, TRADING_DAYS))
    steps = np.exp((mu - 0.5 * sigma * sigma) * DT + sigma * np.sqrt(DT) * z)
    paths = np.empty((n_paths, TRADING_DAYS + 1))
    paths[:, 0] = S0
    paths[:, 1:] = S0 * np.cumprod(steps, axis=1)
    return paths


demo = simulate_paths(4, REGIMES["covid"], seed=1)
print("formes :", {k: simulate_paths(2, v, seed=0).shape for k, v in REGIMES.items()})
print("spot final covid (4 traj.) :", np.round(demo[:, -1], 2))

formes : {'calm': (2, 22), 'covid': (2, 22), 'inflation': (2, 22)}
spot final covid (4 traj.) : [101.17  94.04  95.64  91.1 ]


## 3. Environnement de couverture (gymnasium)

- **Position** : courte de 1 call ATM ($K = S_0$, prime encaissée au départ).
- **État** (5 dimensions, plan du README) : moneyness $S/K - 1$, temps restant $\tau$, delta BS, gamma BS (échelle $\times 10^2$), position de couverture courante $h_{t-1}$.
- **Action** : position $x \in [-1, 1]$, ratio de couverture $a = (x+1)/2 \in [0, 1]$ (reparamétrisation anti-saturation : garder un gradient aux bornes) — on détient $h_t = a_t \cdot \delta_{BS,t}$ parts de sous-jacent. $a = 1$ constant **reproduit exactement** la couverture delta : la baseline est un point spécial de l'espace des politiques, ce qui rend la comparaison lisible.
- **Coûts** : `cost_bps` (points de base) appliqués à la valeur *échangée* $|h_t S_t - h_{t-1} S_{t-1}|$.
- **Récompense** : nulle aux pas intermédiaires, égale au **P&L terminal de couverture** au dernier pas — prime encaissée moins payoff final, plus le P&L cumulé de la couverture, moins les frais. Le P&L est normalisé par $K$ puis remis à l'échelle $\times 100$ (ordre de grandeur $O(1)$, standard pour PPO). L'entraînement PPO joue des épisodes mono-trajectoire ; les exécutions d'évaluation vectorisent.

In [4]:
class HedgingEnv(gym.Env):
    """Couverture d'un short call ATM de 30 jours, rebalancement quotidien."""

    metadata = {"render_modes": []}

    def __init__(self, sigma, n_paths=200_000, seed=0, cost_bps=0.0,
                 episode_batch=1, mu=0.0):
        super().__init__()
        self.sigma = sigma
        self.mu = mu
        self.cost_bps = cost_bps
        self.dt = DT
        self.K = 100.0
        self.n_steps = TRADING_DAYS
        self.episode_batch = episode_batch
        # Banque de trajectoires pre-tirees : episodes servis dans l'ordre.
        self.paths = simulate_paths(n_paths, sigma, seed=seed, mu=self.mu)
        self.cursor = 0
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(5,), dtype=np.float64)
        # Action x dans [-1, 1] ; ratio applique a = (x + 1) / 2.
        # Reparametrisation anti-saturation : clip(mu, 0, 1) en borne laisse un
        # gradient nul (politiques mortes a a=0, mesurees runs precedents) ;
        # l'interieur [-1, 1] garde le gradient vivant aux deux bornes utiles.
        self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float64)

    # -- etat --------------------------------------------------------
    def _batch_obs(self, t, S, h_prev, tau):
        delta = bs_delta(S, self.K, self.sigma, tau)
        gamma = bs_gamma(S, self.K, self.sigma, tau)
        return np.stack([S / self.K - 1.0,
                         np.full_like(S, tau),
                         delta,
                         100.0 * gamma,
                         np.full_like(S, h_prev)], axis=-1)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if self.cursor + self.episode_batch > len(self.paths):
            self.cursor = 0
        sl = slice(self.cursor, self.cursor + self.episode_batch)
        self.cursor += self.episode_batch
        self._S = self.paths[sl].copy()          # (B, n_steps+1)
        self._t = 0
        self._h = np.zeros(self.episode_batch)   # parts detenues
        self._hedge_pnl = np.zeros(self.episode_batch)
        self._costs = np.zeros(self.episode_batch)
        self._premium = bs_call_price(self._S[:, 0], self.K,
                                      self.sigma, self.n_steps * self.dt)
        tau = self.n_steps * self.dt
        return self._batch_obs(0, self._S[:, 0].copy(), 0.0, tau), {}

    def step(self, action):
        a = (np.clip(np.asarray(action).reshape(-1)[0], -1.0, 1.0) + 1.0) / 2.0
        t = self._t
        tau_t = (self.n_steps - t) * self.dt
        S_t = self._S[:, t]
        delta_t = bs_delta(S_t, self.K, self.sigma, tau_t)
        h_new = a * delta_t
        traded = np.abs(h_new * S_t - self._h * S_t)
        self._costs += traded * self.cost_bps / 10_000.0
        # P&L de couverture du pas : h_detenu * (S_{t+1} - S_t)
        S_next = self._S[:, t + 1]
        self._hedge_pnl += self._h * (S_next - S_t)
        self._h = h_new
        self._t += 1
        done = self._t == self.n_steps
        if done:
            S_T = self._S[:, -1]
            payoff = np.maximum(S_T - self.K, 0.0)
            pnl = self._premium - payoff + self._hedge_pnl - self._costs
            reward = float(np.mean(pnl)) / self.K * 100.0  # echelle ~O(1) pour PPO
            obs = self._batch_obs(t + 1, S_T, self._h[-1], 0.0)
            return obs, reward, True, False, {"pnl": pnl}
        tau = (self.n_steps - self._t) * self.dt
        obs = self._batch_obs(self._t, self._S[:, self._t].copy(),
                              float(self._h[0]), tau)
        return obs, 0.0, False, False, {}

### Sanity check

Un épisode batch avec action $a = 1$ constante doit reproduire la couverture delta : P&L centré près de zéro (résidu de rebalancement discret only), sans frais. Et une politique $a = 0$ (pas de couverture) doit donner une dispersion bien plus large — le gauchissement du payoff court.

In [5]:
def run_constant_policy(env, a, n_batches=8):
    """Joue a constant sur n_batches episodes-batch, renvoie les P&L (en $)."""
    pnls = []
    for _ in range(n_batches):
        obs, _ = env.reset()
        done = False
        while not done:
            obs, r, done, _, info = env.step(np.array([a]))
        pnls.append(info["pnl"])
    return np.concatenate(pnls)


env_test = HedgingEnv(REGIMES["calm"], n_paths=4_000, seed=7, episode_batch=500)
pnl_delta = run_constant_policy(env_test, 1.0)
pnl_bare = run_constant_policy(env_test, 0.0)
print(f"delta (a=1) : mean={pnl_delta.mean():+.3f}  std={pnl_delta.std():.3f}  "
      f"cvar95={np.mean(np.sort(pnl_delta)[:int(0.05 * len(pnl_delta))]):+.3f}")
print(f"non couverte (a=0) : mean={pnl_bare.mean():+.3f}  std={pnl_bare.std():.3f}  "
      f"cvar95={np.mean(np.sort(pnl_bare)[:int(0.05 * len(pnl_bare))]):+.3f}")

delta (a=1) : mean=-0.007  std=0.570  cvar95=-1.268
non couverte (a=0) : mean=-0.013  std=1.157  cvar95=-3.266


## 4. Baseline : couverture delta BS quotidienne

Fonction d'évaluation vectorielle (sans RL) : pour chaque trajectoire, $h_t = \delta_{BS,t}$ chaque séance, frais optionnels. Sert de référence à toutes les sections suivantes.

In [6]:
def delta_hedge_pnl(paths, sigma, cost_bps=0.0):
    """P&L de couverture delta quotidienne, vecteur (n_paths,)."""
    n_paths, L = paths.shape
    n_steps = L - 1
    K = 100.0
    premium = bs_call_price(paths[:, 0], K, sigma, n_steps * DT)
    hedge_pnl = np.zeros(n_paths)
    costs = np.zeros(n_paths)
    h = np.zeros(n_paths)
    for t in range(n_steps):
        tau = (n_steps - t) * DT
        delta_t = bs_delta(paths[:, t], K, sigma, tau)
        traded = np.abs(delta_t * paths[:, t] - h * paths[:, t])
        costs += traded * cost_bps / 10_000.0
        hedge_pnl += h * (paths[:, t + 1] - paths[:, t])
        h = delta_t
    payoff = np.maximum(paths[:, -1] - K, 0.0)
    return premium - payoff + hedge_pnl - costs


def cvar95(x):
    k = max(1, int(np.ceil(0.05 * len(x))))
    return float(np.mean(np.sort(x)[:k]))


def summarize(x):
    return {"mean": float(np.mean(x)), "std": float(np.std(x)),
            "cvar95": cvar95(x)}


eval_paths = {k: simulate_paths(2_000, v, seed=100 + i, mu=DRIFT)
              for i, (k, v) in enumerate(REGIMES.items())}
base_rows = []
for k in REGIMES:
    for fees in (0.0, 5.0):
        s = summarize(delta_hedge_pnl(eval_paths[k], REGIMES[k], fees))
        base_rows.append({"regime": k, "fees_bps": fees, "policy": "delta", **s})
pd.DataFrame(base_rows)

,regime,fees_bps,policy,mean,std,cvar95
0,calm,0.0,delta,-0.029864,0.572916,-1.261993
1,calm,5.0,delta,-0.125754,0.569796,-1.367670
2,covid,0.0,delta,-0.074969,1.691379,-3.875349
3,covid,5.0,delta,-0.172875,1.688678,-3.978185
4,inflation,0.0,delta,0.042136,1.194019,-2.597975
5,inflation,5.0,delta,-0.054004,1.192092,-2.700456


## 5. Entraînement PPO — 8 seeds

PPO (stable-baselines3), réseau 64×64, ~120 000 pas par seed sur l'environnement mono-trajectoire (récompense = P&L terminal), entraîné sous **mesure P** (vol d'entraînement 20 % — `HedgingEnv(0.20, …)`, distincte des trois régimes d'évaluation 12/35/25 % de la section 2 ; dérive 8 %/an). Entraînement **sans frais** puis l'évaluation les rajoute : on apprend à couvrir, puis on mesure ce que les frais enlèvent — le même traitement des deux côtés de la comparaison.

In [7]:
import time
from stable_baselines3 import PPO

N_SEEDS = 8
TRAIN_STEPS = 120_000

models = {}
t0 = time.time()
for seed in range(N_SEEDS):
    env = HedgingEnv(0.20, n_paths=400_000, seed=1_000 + seed, episode_batch=1, mu=DRIFT)
    env.reset(seed=seed)
    model = PPO("MlpPolicy", env, seed=seed, gamma=1.0, n_steps=2048,
                batch_size=512, ent_coef=0.0, verbose=0, device="cpu")
    model.learn(total_timesteps=TRAIN_STEPS)
    models[seed] = model
    print(f"seed {seed} : done  ({time.time() - t0:6.1f}s cumules)")
print(f"8 seeds en {time.time() - t0:.1f}s")

seed 0 : done  (  85.7s cumules)


seed 1 : done  ( 171.2s cumules)


seed 2 : done  ( 250.7s cumules)


seed 3 : done  ( 333.1s cumules)


seed 4 : done  ( 418.0s cumules)


seed 5 : done  ( 503.6s cumules)


seed 6 : done  ( 587.2s cumules)


seed 7 : done  ( 667.1s cumules)
8 seeds en 667.1s


## 6. Évaluation : RL contre delta, avec et sans frais

2 000 trajectoires **nouvelles** par régime (graines distinctes de l'entraînement et de la baseline). L'agent joue action = politique ; on rapporte moyenne, écart-type et CVaR 95 % du P&L de couverture. Le tableau agrège les 8 seeds.

In [8]:
def rl_pnl(model, paths, sigma, cost_bps=0.0):
    n_paths, L = paths.shape
    n_steps = L - 1
    K = 100.0
    premium = bs_call_price(paths[:, 0], K, sigma, n_steps * DT)
    hedge_pnl = np.zeros(n_paths)
    costs = np.zeros(n_paths)
    h = np.zeros(n_paths)
    for t in range(n_steps):
        tau = (n_steps - t) * DT
        delta_t = bs_delta(paths[:, t], K, sigma, tau)
        obs = np.stack([paths[:, t] / K - 1.0, np.full(n_paths, tau),
                        delta_t, 100.0 * bs_gamma(paths[:, t], K, sigma, tau),
                        h], axis=-1)
        x = model.predict(obs, deterministic=True)[0].reshape(-1)
        a = (np.clip(x, -1.0, 1.0) + 1.0) / 2.0
        h_new = a * delta_t
        traded = np.abs(h_new * paths[:, t] - h * paths[:, t])
        costs += traded * cost_bps / 10_000.0
        hedge_pnl += h * (paths[:, t + 1] - paths[:, t])
        h = h_new
    payoff = np.maximum(paths[:, -1] - K, 0.0)
    return premium - payoff + hedge_pnl - costs


rows = []
for k, sig in REGIMES.items():
    for fees in (0.0, 5.0):
        base = summarize(delta_hedge_pnl(eval_paths[k], sig, fees))
        rows.append({"regime": k, "fees_bps": fees, "policy": "delta", **base})
        for seed, model in models.items():
            s = summarize(rl_pnl(model, eval_paths[k], sig, fees))
            rows.append({"regime": k, "fees_bps": fees,
                         "policy": f"rl_seed{seed}", **s})
df = pd.DataFrame(rows)
agg = (df.groupby(["regime", "fees_bps", "policy"])[["mean", "std", "cvar95"]]
         .mean().round(4))
agg

mean     std   cvar95
regime    fees_bps policy                           
calm      0.0      delta    -0.0299  0.5729  -1.2620
                   rl_seed0 -0.1203  0.7896  -2.1068
                   rl_seed1 -0.2035  1.0478  -2.3460
                   rl_seed2 -0.3707  1.9755  -5.9440
                   rl_seed3 -0.2766  1.4208  -3.9500
                   rl_seed4 -0.1804  1.0859  -3.2909
                   rl_seed5 -0.0636  0.8190  -2.2811
                   rl_seed6 -0.4075  2.1189  -6.2676
                   rl_seed7 -0.0665  0.9169  -2.6185
          5.0      delta    -0.1258  0.5698  -1.3677
                   rl_seed0 -0.2001  0.7846  -2.1708
                   rl_seed1 -0.2721  1.0554  -2.4136
                   rl_seed2 -0.4174  1.9755  -5.9814
                   rl_seed3 -0.3194  1.4181  -3.9838
                   rl_seed4 -0.2501  1.0816  -3.3450
                   rl_seed5 -0.1735  0.8198  -2.3916
                   rl_seed6 -0.4360  2.1182  -6.2892
                   rl_seed7 -0.1802  0.9194  -2.7419
covid     0.0      delta    -0.0750  1.6914  -3.8753
                   rl_seed0 -0.1069  2.2916  -6.5622
                   rl_seed1 -0.1028  2.3924  -5.8065
                   rl_seed2 -0.1884  6.2612 -18.3765
                   rl_seed3 -0.1381  3.9304 -11.3310
                   rl_seed4 -0.1352  3.3313 -10.6005
                   rl_seed5 -0.1173  2.5336  -7.7200
                   rl_seed6 -0.1848  6.2399 -18.5839
                   rl_seed7 -0.1110  2.8120  -8.5334
          5.0      delta    -0.1729  1.6887  -3.9782
                   rl_seed0 -0.1845  2.2852  -6.6180
                   rl_seed1 -0.1803  2.3950  -5.8758
                   rl_seed2 -0.1988  6.2648 -18.3905
                   rl_seed3 -0.1820  3.9303 -11.3685
                   rl_seed4 -0.1947  3.3250 -10.6403
                   rl_seed5 -0.2194  2.5292  -7.8066
                   rl_seed6 -0.1935  6.2419 -18.5945
                   rl_seed7 -0.2256  2.8115  -8.6558
inflation 0.0      delta     0.0421  1.1940  -2.5980
                   rl_seed0 -0.0312  1.5556  -4.1674
                   rl_seed1 -0.0639  1.7679  -4.1267
                   rl_seed2 -0.2903  4.3037 -12.2133
                   rl_seed3 -0.1589  2.7611  -7.5589
                   rl_seed4 -0.0930  2.1954  -6.6113
                   rl_seed5  0.0104  1.6285  -4.4820
                   rl_seed6 -0.2911  4.3248 -12.3606
                   rl_seed7  0.0169  1.7718  -4.8874
          5.0      delta    -0.0540  1.1921  -2.7005
                   rl_seed0 -0.1090  1.5502  -4.2224
                   rl_seed1 -0.1384  1.7718  -4.1923
                   rl_seed2 -0.3073  4.3060 -12.2294
                   rl_seed3 -0.2014  2.7607  -7.5935
                   rl_seed4 -0.1558  2.1901  -6.6541
                   rl_seed5 -0.0959  1.6269  -4.5799
                   rl_seed6 -0.3030  4.3264 -12.3724
                   rl_seed7 -0.0967  1.7724  -5.0088

## 7. Verdict

Lecture du tableau : comparer chaque régime × frais ligne `delta` contre la moyenne des seeds RL (et la meilleure/pire seed — la dispersion inter-seed dit si le résultat est stable). Un « bat la couverture delta » honnête exige à la fois un écart-type du P&L plus faible **et** une CVaR 95 % moins mauvaise, et de qualifier ce que les frais y changent.

In [9]:
# Tableau de lecture : delta vs RL (moyenne inter-seeds), ecart relatif de std et cvar
verdict_rows = []
for k in REGIMES:
    for fees in (0.0, 5.0):
        sub = df[(df.regime == k) & (df.fees_bps == fees)]
        d = sub[sub.policy == "delta"].iloc[0]
        rl = sub[sub.policy.str.startswith("rl")]
        verdict_rows.append({
            "regime": k, "fees_bps": fees,
            "std_delta": d["std"], "std_rl_mean": rl["std"].mean(),
            "std_rl_min": rl["std"].min(), "std_rl_max": rl["std"].max(),
            "cvar_delta": d["cvar95"], "cvar_rl_mean": rl["cvar95"].mean(),
            "cvar_rl_worst_seed": rl["cvar95"].min(),
        })
vdf = pd.DataFrame(verdict_rows)
vdf["std_reduction_pct"] = (100 * (1 - vdf.std_rl_mean / vdf.std_delta)).round(1)
vdf.round(4)

,regime,fees_bps,std_delta,std_rl_mean,std_rl_min,std_rl_max,cvar_delta,cvar_rl_mean,cvar_rl_worst_seed,std_reduction_pct
0,calm,0.0,0.5729,1.2718,0.7896,2.1189,-1.2620,-3.6006,-6.2676,-122.0
1,calm,5.0,0.5698,1.2716,0.7846,2.1182,-1.3677,-3.6647,-6.2892,-123.2
2,covid,0.0,1.6914,3.7240,2.2916,6.2612,-3.8753,-10.9392,-18.5839,-120.2
3,covid,5.0,1.6887,3.7229,2.2852,6.2648,-3.9782,-10.9937,-18.5945,-120.5
4,inflation,0.0,1.1940,2.5386,1.5556,4.3248,-2.5980,-7.0510,-12.3606,-112.6
5,inflation,5.0,1.1921,2.5381,1.5502,4.3264,-2.7005,-7.1066,-12.3724,-112.9


### Conclusion — verdict NO BEATS

La couverture delta BS quotidienne n'est **battue par aucune des 8 seeds, dans aucun régime, avec ou sans frais** :

- la meilleure seed reste +30 à +38 % au-dessus de l'écart-type du P&L delta selon le régime (calme 12 % : +37,8 % ; 2020 35 % : +35,5 % ; 2022 25 % : +30,3 %) ; la moyenne inter-seeds est ~2,2× ;
- la CVaR 95 % du RL est ~2,8× plus profonde que celle du delta (−3,60 contre −1,26 en calme) ;
- 3 seeds sur 8 restent proches de la politique non couverte (std_rl_max 2,1-6,3) : l'entraînement PPO à récompense terminale est structurellement fragile sur ce problème — mesures aux sections 2 et 3 (mesure neutre sans signal de gradient, saturation du clip aux bornes) ;
- les frais (5 bps) changent ~0,03 % le classement : les deux politiques échangent trop peu pour que les frais décident.

**Ce que le résultat enseigne** (et c'est le point du chapitre 07-01 du livre) : le delta BS est un point spécial très difficile à battre en dispersion ; l'agent RL configuré usine récompensé par le seul P&L terminal ne l'égale pas. La suite naturelle du problème (non couverte ici) : objectif risque-aversion (CVaR/entropique dans la récompense — Kolm & Ritter 2019), ou Q-learning par pas (Halperin 2020).

## 8. Export de la politique pour le portage QC

Le réseau entraîné (seed médiane en écart-type sur le régime central) est exporté en poids numpy dans `rl_policy.py` — passe avant MLP en numpy pur, sans dépendance torch côté QuantConnect.

In [10]:
import torch

# Choix de la seed : celle dont le std RL moyen sur les 3 regimes sans frais est median.
score = {s: df.loc[df.policy == "rl_seed" + str(s), "std"].mean() for s in models}
sorted_seeds = sorted(score, key=score.get)
median_seed = sorted_seeds[len(sorted_seeds) // 2]
print("std moyen par seed :", {s: round(v, 3) for s, v in score.items()})
print("seed mediane retenue pour le portage QC :", median_seed)

model = models[median_seed]
pol = model.policy
linears = [m for m in pol.mlp_extractor.policy_net if isinstance(m, torch.nn.Linear)]
linears.append(pol.action_net)
weights = [(m.weight.detach().numpy().copy(), m.bias.detach().numpy().copy())
           for m in linears]
print("couches :", [w.shape for w, _ in weights])


def _np_arr(a, name):
    return name + " = np.array(" + np.array2string(
        a, separator=", ", max_line_width=100, threshold=10**7) + ")"


out_lines = [
    "# Politique PPO exportee de research.ipynb (#18902), section 8.",
    "# MLP numpy pur (pas de torch) : tanh(64) -> tanh(64) -> mu, ratio = (clip(mu, -1, 1) + 1) / 2.",
    "# Seed mediane " + str(median_seed) + ", PPO 120 000 pas, gamma=1, sans frais (reproductible).",
    "import numpy as np",
    "",
]
for i, (w, b) in enumerate(weights, 1):
    out_lines.append(_np_arr(w, "W" + str(i)))
    out_lines.append(_np_arr(b, "b" + str(i)))
    out_lines.append("")
out_lines += [
    "class RLPolicy:",
    "    # Passe avant deterministe : sortie brute x dans [-1, 1],",
    "    # ratio de couverture a = (x + 1) / 2 dans [0, 1] (section 3).",
    "",
    "    @staticmethod",
    "    def act(state):",
    "        x = np.tanh(state @ W1.T + b1)",
    "        x = np.tanh(x @ W2.T + b2)",
    "        mu = x @ W3.T + b3",
    "        return float((np.clip(mu, -1.0, 1.0) + 1.0) / 2.0)",
    "",
]
with open("rl_policy.py", "w", encoding="utf-8", newline="\n") as f:
    f.write("\n".join(out_lines))
print("rl_policy.py ecrit,", len(weights), "couches")

std moyen par seed : {0: np.float64(1.543), 1: np.float64(1.738), 2: np.float64(4.181), 3: np.float64(2.704), 4: np.float64(2.202), 5: np.float64(1.659), 6: np.float64(4.228), 7: np.float64(1.834)}
seed mediane retenue pour le portage QC : 4
couches : [(64, 5), (64, 64), (1, 64)]
rl_policy.py ecrit, 3 couches
